In [1]:
import GradientGang.Pipeline.Optimizer.OptunaOptimizer
from GradientGang.Pipeline.Optimizer.OptunaOptimizer import OptunaOptimizer

In [ ]:
data_params_s = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.1,
    'shuffle': True,
}
data_params_d = {"stage": "fit", "includeTestInTrain": False}
architecture = {
    "arch_type": "direct",
    "LearningRate": None, # 1e-3, 1e-5
    "Patience": 5,
    #"ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
    "RegularizationWeight": None,   # 0.1, 1e-2
    "EncoderParams": {
        "activation_function": None, # LeakyRelu, GeLU, ReLU
        "layer_type": [
            {
                "name": "LSTM",
                "params": {
                    "input_size": 34,
                    "hidden_size": None,    # categ [64, 128, 256]
                    "num_layers": None,     # 1-3
                    "bias": True,
                    "batch_first": True,
                    "dropout": None,        # 0, 0.5
                    "bidirectional": None   # categ [True, False]
                }
            },
            {
                "name": "Linear",
                "params": {
                    "in_features": None,    # constraint = LSTM_hidden O hidden * 2 se bidirectional
                    "out_features": None,   # categ [64, 128]
                    "bias": True,
                }
            }
        ]
    },
    "GlobalFFEncoderParams": {
        "activation_function": "LeakyReLU",
            "layer_type": [
                {
                    "name": "Linear",
                    "params": {
                        "in_features": 1,
                        "out_features": 1,
                        "bias": True,
                    }
                }
            ]
    },

    "FeedForwardParams": {
            "activation_function": None,    # categ [ReLU, GeLU, ...]
            "layer_type": [
                {
                    "name": "Linear",
                    "params": {
                        "in_features": None,    # out feature di Linear Encoder + 1
                        "out_features": None,   # categ [16, 32]
                        "bias": True,
                    }
                }
            ]
        },

    "OutputDim": 3  # Number of classes
}
hyper_arch = [
    {
        "name": "learning_rate",
        "type": "float",
        "path": ["LearningRate"],
        "opts": {"low": 1e-3, "high": 1e-5, "log": True}
    },
    {
        "name": "regularization",
        "type": "float",
        "path": ["RegularizationWeight"],
        "opts": {"low": 0.1, "high": 1e-2 , "log": True}
    },
    {
        "name": "encoder_activation",
        "type": "categ",
        "path": ["EncoderParams", "activation_function"],
        "opts": {"choices": ["LeakyReLU", "GeLU", "ReLU"]}
    },
    {
        "name": "LSTM_hidden",
        "type": "categ",
        "path": ["EncoderParams", "layer_type", 0, "params", "hidden_size"],
        "opts": {"choices": [64, 128, 256]}
    },
    {
        "name": "LSTM_dropout",
        "type": "float",
        "path": ["EncoderParams", "layer_type", 0, "params", "dropout"],
        "opts": {"low": 0.0, "high": 0.5}
    },
    {
        "name": "LSTM_bidir",
        "type": "categ",
        "path": ["EncoderParams", "layer_type", 0, "params", "bidirectional"],
        "opts": {"choices": [True, False]}
    },
    {
        "name": "Linear_in",
        "type": "int",
        "path": ["EncoderParams", "layer_type", 1, "params", "in_features"],
        "opts": {"choices": [True, False]}
    }
]
constr_arch = []
hyper_data = [
    {
        "name": "stage",
        "type": "value",
        "paths": [[]],
        "opts": {"value": "fit"}
    },
    {
        "name": "includeTestInTrain",
        "type": "value",
        "paths": [[]],
        "opts": {"value": False}
    }
]
constr_data = []

In [ ]:
params = {"dataloader": data_params_s, "hyper_dataloader": hyper_data, "arch": architecture, "hyper_arch": hyper_arch,
          "constr_arch": constr_arch, "constr_dataloader": constr_data}

In [4]:
optimizer = OptunaOptimizer(params)

In [6]:
optimizer.optimize(5)

[I 2025-11-11 12:54:37,961] A new study created in memory with name: no-name-7c53dd6f-8727-436c-b81d-d5f6534f459f
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name             | Type              | Params | Mode 
---------------------------------------------------------------
0 | encoder          | Encoder           | 455 K  | train
1 | globalff_encoder | FeedForward       | 2      | train
2 | feedforward      | FeedForward       | 10.5 K | train
3 | val_f1           | MulticlassF1Score | 0      | train
---------------------------------------------------------------
465 K     Trainable params
0         Non-trainable params
465 K     Total params
1.863     Total estimated model params size (MB)
19        Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  9.24it/s]

c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (19) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 4: 100%|██████████| 19/19 [00:07<00:00,  2.50it/s, v_num=15, val_F1=0.749]

...

c:\Users\teopa\Documents\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\teopa\Documents\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\teopa\Documents\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (19) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 4: 100%|██████████| 19/19 [00:07<00:00,  2.51it/s, v_num=19, val_F1=0.141]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 19/19 [00:07<00:00,  2.51it/s, v_num=19, val_F1=0.141]

[I 2025-11-11 12:58:02,259] Trial 4 finished with value: 0.14095500111579895 and parameters: {'includeTestInTrain': False, 'num_layers': 1, 'dropout': 0.8557351336396671, 'batch_first': True}. Best is trial 2 with value: 0.14095500111579895.
